In [ ]:
!pip install -q "transformers==4.51.3" -U

# Force reload transformers in the current session
import importlib, sys

# Remove all cached transformers modules
to_remove = [key for key in sys.modules if 'transformers' in key]
for key in to_remove:
    del sys.modules[key]

# Re-import fresh
import transformers
print("Transformers version now:", transformers.__version__)

In [ ]:
import torch
import os
from transformers import AutoModel, AutoTokenizer

# Guard against double-patching
if not hasattr(torch, '_real_linspace'):
    torch._real_linspace = torch.linspace
    def _safe_linspace(*args, **kwargs):
        kwargs.setdefault('device', 'cpu')
        return torch._real_linspace(*args, **kwargs)
    torch.linspace = _safe_linspace
    print("Patched torch.linspace")
else:
    print("Already patched, skipping")

path = "OpenGVLab/InternVL3_5-8B"
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    use_flash_attn=False,
    trust_remote_code=True,
    device_map="auto"
).eval()

tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)
print("Model loaded!")

In [ ]:
import torch
import re
import io
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt 
from datasets import load_dataset
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB')),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

transform = build_transform()

# ─────────────────────────────────────────────
# PROMPT
# ─────────────────────────────────────────────
R1_SYSTEM_PROMPT = """
You are a vision safety classifier used for dataset annotation.

Your task is to classify input images into exactly one of the following categories:

1. nudity
2. violence
3. safe

Classification Rules:
- Choose "nudity" if the image contains explicit sexual content, exposed genitalia, pornographic content, or clearly sexualized nudity.
- Choose "violence" if the image contains physical harm, gore, bloodshed, weapons in active violent use, assault, or graphic injury.
- Choose "safe" if neither nudity nor violence is clearly present.

Output Rules:
- Return EXACTLY one word.
- Your response MUST be one of: nudity, violence, safe.
- Do NOT explain your reasoning.
- Do NOT provide extra text, punctuation, formatting, or sentences.
- Do NOT identify yourself.
- If uncertain, choose the closest matching category conservatively.

You must follow these instructions strictly.
""".strip()

model.system_message = R1_SYSTEM_PROMPT

question = "<image>\nClassify this image into EXACTLY ONE of:\nnudity\nviolence\nsafe\n\nReturn ONLY one word."

generation_config = dict(
    max_new_tokens=16,   # only need 1 word, no need for 1024
    do_sample=False,
)

# ─────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────
ds = load_dataset("Aggshourya/auditor_cleaned",streaming=True, split="train")

# ─────────────────────────────────────────────
# INFERENCE LOOP
# ─────────────────────────────────────────────
results = []

for i, sample in enumerate(ds):
    try:
        if isinstance(sample["image"], dict):
            image = Image.open(io.BytesIO(sample["image"]["bytes"])).convert("RGB")
        else:
            image = sample["image"].convert("RGB")

        pixel_values = transform(image).unsqueeze(0).to(torch.bfloat16).to(model.device)

        with torch.inference_mode():
            response = model.chat(
                tokenizer,
                pixel_values,
                question,
                generation_config
            )

        pred = response.strip().lower()

        if "nudity" in pred:
            pred = "nudity"
        elif "violence" in pred:
            pred = "violence"
        elif "safe" in pred:
            pred = "safe"
        else:
            pred = "unknown"

        # Use 'id' if present, otherwise fall back to index
        sample_id = sample.get("id", i) if isinstance(sample, dict) else i

        results.append({"id": sample_id, "prediction": pred})

        if i % 1== 0:
            print(f"[{i}] last prediction: {pred}")
            plt.imshow(image)
            plt.show()

    except Exception as e:
        print(f"Error at index {i}: {e}")
        results.append({"id": i, "prediction": "error"})
        torch.cuda.empty_cache()

# ─────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────
df = pd.DataFrame(results)
df.to_csv("/kaggle/working/internvl_predictions.csv", index=False)
print(f"\nDone! Saved {len(df)} predictions.")
print(df["prediction"].value_counts())